# Laboratorio de Robot de entregas en un almacenamiento

A partir de la **imagen**, construye el MDP y resuÃ©lvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convencion y notacion

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas Ã— 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterÃ­as / paredes;
- zona de entrega `+10` (**terminal**);
- estaciÃ³n de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convenciÃ³n del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### DinÃ¡mica

La transiciÃ³n depende del **estado actual**:

**Piso normal**

$$
P(\text{direcciÃ³n elegida})=0.90
$$

$$
P(\text{desviaciÃ³n izquierda})=0.05
$$

$$
P(\text{desviaciÃ³n derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{direcciÃ³n elegida})=0.60
$$

$$
P(\text{desviaciÃ³n izquierda})=0.20
$$

$$
P(\text{desviaciÃ³n derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estanterÃ­a, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 â€” Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas lÃ­neas de cÃ³digo: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstÃ¡culos;
- tipos de piso;
- funciÃ³n de transiciÃ³n.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6
        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}
        self.terminal_states = {
            (0, 5): 10.0,  # Entrega
            (2, 2): 2.0,   # Estacion de carga
            (3, 5): -10.0, # Peligro mortal
        }
        self.danger_states = {(1, 4): -3.0, (4, 1): -3.0}
        self.living_reward = -1.0
        self.gamma = 0.9
        self.actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    def is_valid_state(self, state):
        row, col = state
        return 0 <= row < self.height and 0 <= col < self.width and state not in self.walls

    def states(self):
        return [(r, c) for r in range(self.height) for c in range(self.width)
                if self.is_valid_state((r, c))]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        return self.danger_states.get(state, self.living_reward)

    def get_transition_probs(self, state, action):
        if not self.is_valid_state(state):
            raise ValueError(f"Estado invalido: {state}")
        if action not in self.actions:
            raise ValueError(f"Accion invalida: {action}")
        if self.is_terminal(state):
            return [(state, 1.0)]

        intended, slip = (0.60, 0.20) if state in self.slippery_states else (0.90, 0.05)
        left = (action[1], -action[0])
        right = (-action[1], action[0])
        probabilities = {}
        for (dr, dc), probability in ((action, intended), (left, slip), (right, slip)):
            next_state = (state[0] + dr, state[1] + dc)
            if not self.is_valid_state(next_state):
                next_state = state
            probabilities[next_state] = probabilities.get(next_state, 0.0) + probability
        return list(probabilities.items())


### ValidaciÃ³n mÃ­nima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("NÃºmero de estados:", len(S))

# Cada distribuciÃ³n T(s,a,Â·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("âœ“ Todas las distribuciones de transiciÃ³n suman 1.")


NÃºmero de estados: 26
âœ“ Todas las distribuciones de transiciÃ³n suman 1.



## Parte 2 â€” Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    return sum(p * V[next_state] for next_state, p in grid.get_transition_probs(state, action))


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    states = grid.states()
    V = {state: 0.0 for state in states}
    for state in grid.terminal_states:
        V[state] = grid.get_reward(state)

    for iteration in range(1, max_iter + 1):
        updated = V.copy()
        delta = 0.0
        for state in states:
            if grid.is_terminal(state):
                continue
            best = max(expected_next_value(grid, state, action, V) for action in grid.actions)
            updated[state] = grid.get_reward(state) + grid.gamma * best
            delta = max(delta, abs(updated[state] - V[state]))
        V = updated
        if delta < threshold:
            return V, iteration
    raise RuntimeError(f"Value Iteration no convergio en {max_iter} iteraciones")


def extract_policy(grid, V):
    return {
        state: max(grid.actions, key=lambda action: expected_next_value(grid, state, action, V))
        for state in grid.states() if not grid.is_terminal(state)
    }


## Parte 3 â€” Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    states = grid.states()
    V = {state: 0.0 for state in states}
    for state in grid.terminal_states:
        V[state] = grid.get_reward(state)

    for _ in range(max_iter):
        updated = V.copy()
        delta = 0.0
        for state in states:
            if grid.is_terminal(state):
                continue
            updated[state] = grid.get_reward(state) + grid.gamma * expected_next_value(
                grid, state, policy[state], V
            )
            delta = max(delta, abs(updated[state] - V[state]))
        V = updated
        if delta < threshold:
            return V
    raise RuntimeError(f"Policy Evaluation no convergio en {max_iter} iteraciones")


def policy_improvement(grid, V):
    return extract_policy(grid, V)


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    states = [s for s in grid.states() if not grid.is_terminal(s)]
    policy = {s: grid.actions[0] for s in states}
    history = []
    for _ in range(max_iter):
        V = policy_evaluation(grid, policy, threshold=threshold)
        improved = policy_improvement(grid, V)
        changed = sum(policy[s] != improved[s] for s in states)
        history.append(changed)
        policy = improved
        if changed == 0:
            return policy, V, history
    raise RuntimeError(f"Policy Iteration no convergio en {max_iter} iteraciones")


## Parte 4 â€” VisualizaciÃ³n y comparaciÃ³n


In [5]:
ARROWS = {
    (-1, 0): "â†‘",
    ( 1, 0): "â†“",
    ( 0,-1): "â†",
    ( 0, 1): "â†’",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolÃ­tica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolÃ­tica:")
print_policy(grid, pi_pi)

# Comprobación más robusta (revisa si los valores esperados son idénticos)
max_diff = max(abs(V_vi[s] - V_pi[s]) for s in grid.states())
assert max_diff < 1e-3, f"Los valores no coinciden: diferencia máxima {max_diff}"
print(f"✓ Ambos algoritmos convergieron al mismo valor (diferencia máx: {max_diff:.6f}).")


=== VALUE ITERATION ===
Iteraciones: 19

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

PolÃ­tica:
 â†’  |  â†’  |  â†“  |  #  |  â†’  | +10
 â†“  |  #  |  â†“  |  â†’  |  â†’  |  â†‘ 
 â†’  |  â†’  | +2 |  â†‘  |  #  |  â†‘ 
 â†’  |  â†’  |  â†‘  |  â†‘  |  â†  | -10
 â†‘  |  â†‘  |  #  |  â†‘  |  â†  |  â† 

=== POLICY ITERATION ===
Historia: [16, 5, 1, 0]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

PolÃ­tica:
 â†’  |  â†’  |  â†“  |  #  |  â†’  | +10
 â†“  |  #  |  â†“  |

## Parte 5 ? Interpreta la pol?tica

1. **Con los par?metros originales, el robot prefiere la estaci?n de carga +2.** La ruta m?s probable desde START es (0,0) ? (0,1) ? (0,2) ? (1,2) ? (2,2).
2. La recompensa menor puede ser ?ptima porque se obtiene antes y con menos pasos y riesgo. La entrega +10 est? m?s lejos; el costo por paso, el descuento y los peligros pueden hacer que su retorno esperado sea menor.
3. En (1,2), que es resbaloso, la pol?tica elige bajar hacia la carga; si esa celda fuera normal, elegir?a ir a la derecha para rodear y buscar la entrega. En (2,1) y (3,3) la incertidumbre cambia los retornos, aunque en el caso base no cambia la acci?n ?ptima. El efecto tambi?n puede propagarse a estados vecinos.
4. El -1 representa el costo de cada paso. Favorece rutas cortas y reduce el valor de recompensas lejanas que requieren muchos movimientos.
5. No se puede usar una ?nica distribuci?n para todo el grid porque las probabilidades dependen del tipo de piso del **estado actual**: en piso normal son 0.90/0.05/0.05 y en piso resbaloso son 0.60/0.20/0.20.

### Experimento A ? Menos costo por paso

**Predicci?n:** con living_reward = -0.1, el robot tolera una ruta m?s larga para alcanzar la entrega. Al ejecutar, la ruta m?s probable termina en +10.

### Experimento B ? Piso muy resbaloso

Con movimiento deseado 0.40 y desviaciones 0.30 cada una, la ruta esperada hacia la entrega se vuelve m?s riesgosa. En este mapa el robot mantiene su preferencia por carga +2.

### Experimento C ? M?s paciencia

Con gamma = 0.99, las recompensas futuras se descuentan menos. La pol?tica desde START cambia y busca la entrega +10.

### Bonus

Con gamma = 0.9 y las dem?s reglas originales, el cambio desde carga hacia entrega ocurre aproximadamente cuando living_reward sube de -0.80 (umbral observado cerca de -0.798). El valor exacto depende de la tolerancia num?rica y de la convenci?n de recompensa terminal.
